In [0]:
vol_landing_path="/Volumes/rag_on_databricks/landing/vol_landing/Orion_Docs/"
dbutils.fs.ls(vol_landing_path)

In [0]:
from pyspark.sql.functions import expr

# Read all files from the documents volume
docs_df = spark.read.format("binaryFile").load(vol_landing_path)

# Parse each document using ai_parse_document (use expr to call the SQL AI function)
parsed_df = docs_df.withColumn("parsed_content", 
                           expr(f"""ai_parse_document(content, map(
                                "version", "2.0",
                                "imageOutputPath", "{vol_landing_path}parsed_images/"
                               ))""")
                          )
# Drop binary content
parsed_df = parsed_df.drop("content")

# Display a sample of the parsed results
display(parsed_df)


In [0]:
# Import the DocumentRenderer helper class
import sys
import os

# Add parent directory to path FIRST
sys.path.append(os.path.abspath('..'))

# NOW import from Includes (after it's in the path)
from Includes.document_renderer import render_ai_parse_output, render_ai_parse_output_interactive

In [0]:
# Select a sample document and render its parsed content using render_ai_parse_output
sample = parsed_df.select("parsed_content").limit(1).collect()

if sample:
    doc = sample[0]["parsed_content"]
    render_ai_parse_output(doc)
else:
    print("No parsed documents found. Please check your input volume and parsing step.")